## Registering COGs as Earth Engine Assets

Created by Mel Rose; Last updated 9/8/2026

## Setup

In [ ]:
ee_project = 'landandcarbon'
bucket_name = 'wri-lcl-wvsc' #change to bucket name where COGs are stored
directory = 'LATAM_WVSC' #directory within bucket where files are stored


In [ ]:
from google.colab import auth
from google.cloud import storage

import ee
import json
from pprint import pprint
import datetime
from google.auth.transport.requests import AuthorizedSession

auth.authenticate_user()
ee.Initialize(project=ee_project)

session = AuthorizedSession(
    ee.data.get_persistent_credentials().with_quota_project(ee_project)
)

In [ ]:
#List COGs in bucket
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucket_name)

blobs = bucket.list_blobs(prefix=directory)

tif_files = [blob.name.split('/')[-1] for blob in blobs if blob.name.endswith('.tif')]

tif_files

['LATAM_V1_Change_2015.tif', 'LATAM_V1_TCC_2015.tif', 'LATAM_V1_TCH_2015.tif']

In [ ]:
#Function to execute COG registration request
def register_cog_as_ee_asset(session, request, ee_project):
  url = f'https://earthengine.googleapis.com/v1alpha/projects/{ee_project}/image:importExternal'

  response = session.post(
    url = url,
    data = json.dumps(request)
  )

  return json.loads(response.content)

## Register a single COG

Information on image manifest and registering COGs as assets:

*   https://developers.google.com/earth-engine/guides/image_manifest
*   https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff



In [ ]:
#Define request
request = {
  'imageManifest': {
    'name': f'projects/{ee_project}/assets/LATAM_WVSC/V1_Change_2015', #Edit folder and asset name to the EE folder and asset name you want within your ee project. This will be the asset ID.
    'tilesets': [
      { 'sources': [ { 'uris': [f'gs://{bucket_name}/{directory}/{tif_files[0]}'] } ] } #tif_file to the name of the COG you want to register.
    ],
  },
}

request

In [ ]:
#Execute COG registration request
result = register_cog_as_ee_asset(session, request, ee_project)
result

## Combine multiple files into a single image with multiple bands

The following manifest describes how to combine muliple COG files into a single multiband image.

In [ ]:
#Define request
#Note: the parent folder must already exist. Create it in the EE project first before registering the asset.
request = {
  'imageManifest': {
    'name': f'projects/{ee_project}/assets/WVSC/LATAM_V1/2015', #Edit folder and asset name to the EE folder and asset name you want within your ee project. This will be the asset ID.
    'uriPrefix': f'gs://{bucket_name}/{directory}/',
    'tilesets': [
      { 'id': '0', 'sources': [ { 'uris': ['LATAM_V1_TCC_2015.tif'] } ] }, #Edit URI to the files you want for each band
      { 'id': '1', 'sources': [ { 'uris': ['LATAM_V1_TCH_2015.tif'] } ] },
      { 'id': '2', 'sources': [ { 'uris': ['LATAM_V1_Change_2015.tif'] } ] },
    ],
    'bands': [
      { 'id': 'TCC', 'tilesetId': '0' }, #Change the id to your desired band name, tilesetID should match the id of the tileset of that band, defined above
      { 'id': 'TCH', 'tilesetId': '1' },
      { 'id': 'Change', 'tilesetId': '2' },

    ],
    'startTime': '2015-01-01T00:00:00.000000000Z',
    'endTime': '2016-01-01T00:00:00.000000000Z',
  },
}

request

{'imageManifest': {'name': 'projects/ee-simsjmichelle/assets/WVSC/LATAM_V1/2015',
  'uriPrefix': 'gs://wri-lcl-wvsc/LATAM_WVSC/',
  'tilesets': [{'id': '0', 'sources': [{'uris': ['LATAM_V1_TCC_2015.tif']}]},
   {'id': '1', 'sources': [{'uris': ['LATAM_V1_TCH_2015.tif']}]},
   {'id': '2', 'sources': [{'uris': ['LATAM_V1_Change_2015.tif']}]}],
  'bands': [{'id': 'TCC', 'tilesetId': '0'},
   {'id': 'TCH', 'tilesetId': '1'},
   {'id': 'Change', 'tilesetId': '2'}],
  'startTime': '2015-01-01T00:00:00.000000000Z',
  'endTime': '2016-01-01T00:00:00.000000000Z'}}

In [ ]:
result = register_cog_as_ee_asset(session, request, ee_project)
result

{}

# Register images as `ee.ImageCollection` time series

The code below creates an image collection, where each image is structured according to the format above and an image is created for each year.

In [ ]:
#Function to create image collection asset container
def create_image_collection(session, ee_project, asset_id, start_time=None, end_time=None, properties=None):
    """Create an ImageCollection asset container in Earth Engine."""
    if properties is None:
        properties = {}

    request = {
        'type': 'IMAGE_COLLECTION',
        'properties': properties,
    }

    if start_time:
        request['startTime'] = start_time
    if end_time:
        request['endTime'] = end_time

    url = f'https://earthengine.googleapis.com/v1alpha/projects/{ee_project}/assets?assetId={asset_id}'

    response = session.post(
        url=url,
        headers={'Content-Type': 'application/json'},
        data=json.dumps(request)
    )

    return json.loads(response.content)

In [ ]:
#Create image collection asset container
#Note: the parent folder must already exist. Create it in the EE project first before registering the asset. In this example it would be WVSC
asset_id = 'WVSC/LATAM_V1' #Adjust based on your desired asset ID for image collection
#start and end time of entire image collection
start_time, end_time = '2015-01-01T00:00:00.000000000Z', '2025-01-01T00:00:00.000000000Z' #Adjust based on start/end time of image collection

create_image_collection(session, ee_project, asset_id, start_time, end_time)

{'type': 'IMAGE_COLLECTION',
 'name': 'projects/ee-simsjmichelle/assets/WVSC/LATAM_V1',
 'id': 'projects/ee-simsjmichelle/assets/WVSC/LATAM_V1',
 'updateTime': '2025-11-26T19:37:08.817972Z',
 'startTime': '2015-01-01T00:00:00Z',
 'endTime': '2016-01-01T00:00:00Z'}

In [ ]:
#Extract unique dates from TIFF filenames
dates = [file_name.split('_')[-1].replace('.tif', '') for file_name in tif_files]
unique_dates = sorted(list(set(dates)))
unique_dates

['2015']

In [ ]:
# Define band-to-filename pattern (matching filename structure)
band_map = {
    'TCC': 'LATAM_V1_TCC_{date}.tif',
    'TCH': 'LATAM_V1_TCH_{date}.tif',
    'Change': 'LATAM_V1_Change_{date}.tif'
}

In [ ]:
# Iterate through each unique date and register the COG-backed asset
for date in unique_dates:

    tilesets, bands = [], []
    for i, (band_id, pattern) in enumerate(band_map.items()):
        tilesets.append({
            'id': str(i),
            'sources': [{'uris': [pattern.format(date=date)]}]
        })
        bands.append({
            'id': band_id,
            'tilesetId': str(i)
        })
    year = int(date)
    request = {
        'imageManifest': {
            'name': f'projects/{ee_project}/assets/WVSC/LATAM_V1/{date}',
            'uriPrefix': f'gs://{bucket_name}/{directory}/',
            'tilesets': tilesets,
            'bands': bands,
            'startTime': f'{year}-01-01T00:00:00.000000000Z',
            'endTime': f'{year + 1}-01-01T00:00:00.000000000Z',
            'properties': {'year': year}
        }
    }
    result = register_cog_as_ee_asset(session, request, ee_project)
    display(date, result)

'2015'

{}